# 🚀 Adaptive Residual Steering (ARS) — Kaggle Execution Notebook

This notebook runs the complete, reproducible **Adaptive Residual Steering (ARS)** research pipeline on Kaggle GPU environments (T4 x2, P100, or A100).

### Pipeline Structure
1. **Environment Setup & Verification**
2. **Repository Clone & Package Import**
3. **Model & Hook Initialization (Phi-2 Backbone)**
4. **Stage A: Frozen Baseline Evaluation (GSM8K Benchmark)**
5. **Phase 1.5: Router Bootstrap (Dynamic Layer Selection Training via RLOO)**
6. **Stage C / Multi-Shot Benchmark Evaluation (0, 2, 4, 8 Shot Comparison)**
7. **Export Checkpoints & Performance Metrics**

## 1. Environment Setup & Hardware Check

In [ ]:
!nvidia-smi
!pip install -q transformers>=4.36.0 datasets>=2.16.0 accelerate>=0.25.0

## 2. Clone Repository or Set Workspace Path

In [ ]:
import os
import sys

# If running directly inside cloned repo on Kaggle:
REPO_DIR = "/kaggle/working/adaptive-residual-steering"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Abdullah182155/adaptive-residual-steering.git
    %cd adaptive-residual-steering
else:
    %cd {REPO_DIR}

sys.path.append(os.path.abspath("."))
print(f"Active Working Directory: {os.getcwd()}")

## 3. Load Frozen Backbone Model & ARS Modules

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC
from src.data.dataset import gsm8k_to_cot
from src.evaluation.benchmark import eval_gsm8k_benchmark
from src.training.trainer import train_router_bootstrap

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

CFG = RSCConfig()
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
    trust_remote_code=True,
).to(DEVICE)

model = Phi2WithRSC(base_model, CFG, device=DEVICE)
print(f"Wrapped Candidate Layers: {model.target_layers} (Max active K: {CFG.max_active_layers})")

## 4. Stage A: Frozen Baseline Evaluation (Pre-Steering)

In [ ]:
# Disable any routing intervention to measure pure frozen baseline performance
model.set_routing_override("none")
res_baseline = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage A - Frozen Baseline",
    n_test=50,
    n_shot=0,
    display_examples=3,
)
model.set_routing_override(None)

## 5. Phase 1.5: Router Bootstrap (Dynamic Layer Selection Training via RLOO)

In [ ]:
print("Loading GSM8K training CoT dataset...")
train_data, eval_data = gsm8k_to_cot(tokenizer, CFG)

# Configure epochs and examples for Kaggle execution
CFG.router_bootstrap_epochs = 3
CFG.router_bootstrap_n_examples = 1000

print("Training Joint Subset Router via RLOO Counterfactual Loss...")
best_state, best_loss = train_router_bootstrap(model, train_data, eval_data, CFG, DEVICE)
print(f"Router Bootstrap Complete. Best Probe Loss: {best_loss:.4f}")

## 6. Stage C: Full ARS Benchmark Evaluation (Learned Routing)

In [ ]:
model.set_routing_override(None)  # Dynamic learned routing
model.eval()

res_ars = eval_gsm8k_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    stage_name="Stage C - Full ARS (Learned Router)",
    n_test=50,
    n_shot=0,
    display_examples=3,
)

print("\n===== COMPARISON SUMMARY =====")
print(f"Stage A (Frozen Baseline): {res_baseline['summary']['accuracy']:.2f}%")
print(f"Stage C (Full ARS Router): {res_ars['summary']['accuracy']:.2f}%")

## 7. Save Artifacts & Generated Checkpoints

In [ ]:
import json

summary_export = {
    "stage_a_baseline": res_baseline["summary"],
    "stage_c_ars": res_ars["summary"],
}

with open("kaggle_experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_export, f, indent=2)

print("Experiment results saved to kaggle_experiment_summary.json")